# PyTorch Simple Image Classifier

This notebook shows a simplified PyTorch version of the same image classification task.

The main ideas are the same as the Keras version:
- load image data,
- build a small CNN,
- train the model,
- and evaluate the result.

This version is easier to understand for learners who want to see the PyTorch training loop.

## 1. Import libraries

We import PyTorch, torchvision, and standard Python tools.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18

print("PyTorch is ready")

## 2. Set dataset path and transforms

The dataset is stored in folders by class name. `ImageFolder` automatically reads the labels.

In [ ]:
DATASET_PATH = os.path.join('.', 'images_dataSAT')

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
print("Classes:", full_dataset.classes)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

## 3. Build a small CNN in PyTorch

This network learns image features with convolution layers and classifies them with a final output layer.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

## 4. Train the model

The training loop updates model weights based on the loss from each batch.

In [ ]:
for epoch in range(5):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        labels = labels.float().view(-1, 1)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch + 1} - loss: {running_loss / len(train_loader):.4f}")

## 5. Evaluate the model

We run the model on validation data and calculate accuracy.

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        labels = labels.float().view(-1, 1)
        outputs = model(images)
        preds = (outputs >= 0.5).float()
        total += labels.size(0)
        correct += (preds == labels).sum().item()

accuracy = correct / total
print(f"Validation accuracy: {accuracy:.4f}")

## Summary

This PyTorch notebook follows the same structure as the Keras version:
1. prepare the dataset,
2. create a CNN,
3. train with a loss function and optimizer,
4. evaluate accuracy.

PyTorch gives a more explicit training loop, while Keras gives a more compact high-level API.